# Kubernetes Security Drift Intelligence
## Using Attack Path Analysis and Rule-Based Risk Assessment
**Project Progress – Phase 2 (Design and Implementation) · Python implementation**

This notebook contains the complete Python implementation, organised in the same `app/` package layout as the project repository.
Each code cell writes one module to disk with `%%writefile`, so the notebook can be run top-to-bottom in a fresh Colab session.

**Pipeline implemented so far**

`Kubernetes cluster → Resource collection → Snapshot (normalize + SHA-256) → Baseline → Drift detection → Rule-based evaluation → Impact score`

**How to run:** `Runtime ▸ Run all`.
Colab cannot reach a local cluster (the project was developed against a local `kind` / Docker Desktop cluster), so section 8 runs the pipeline in **demo mode** on the saved baseline/current snapshots from the repository. The live-cluster entry point (`app/main.py`) is included unchanged for use on a machine with a kubeconfig.

## 1. Setup

In [1]:
# Install dependencies (Kubernetes client for the cluster module, PyYAML for the rules file)
!pip install -q kubernetes pyyaml

import os
for d in ["app", "rules", "snapshots"]:
    os.makedirs(d, exist_ok=True)
print("Project folders ready:", os.listdir("."))

Project folders ready: ['app', 'snapshots', 'K8s_Security_Drift_Intelligence_Phase2.ipynb', 'rules']


## 2. Project structure
```
app/
├── __init__.py
├── resources.py   # KubernetesResource, Pod, Service, NetworkPolicy
├── cluster.py     # KubernetesCluster (Kubernetes API access)
├── snapshot.py    # ConfigurationSnapshot
├── baseline.py    # BaselineManager
├── drift.py       # DriftEvent, DriftDetectionEngine
├── rules.py       # SecurityRuleEngine
└── main.py        # end-to-end entry point (live cluster)
rules/security_rules.yaml
snapshots/baseline.json, current.json
```

## 3. Resource model & collection

In [2]:
# Marks `app` as a Python package
open('app/__init__.py', 'w').close()
print('app/__init__.py created')

app/__init__.py created


### 3.1 `app/resources.py`
Resource model: `KubernetesResource` base dataclass with `Pod`, `Service` and `NetworkPolicy` subclasses. Each subclass converts a Kubernetes API object with `from_api_object()`.

In [3]:
%%writefile app/resources.py
from dataclasses import dataclass
from typing import Any


@dataclass
class KubernetesResource:
    kind: str
    name: str
    namespace: str
    data: dict[str, Any]


@dataclass
class Pod(KubernetesResource):
    privileged: bool = False
    run_as_root: bool = False

    @classmethod
    def from_api_object(cls, pod):

        privileged = False
        run_as_root = False

        for container in pod.spec.containers:

            security_context = container.security_context

            if security_context:

                if security_context.privileged:
                    privileged = True

                if security_context.run_as_user == 0:
                    run_as_root = True

        return cls(
            kind="Pod",
            name=pod.metadata.name,
            namespace=pod.metadata.namespace,
            data={
                "labels": pod.metadata.labels or {}
            },
            privileged=privileged,
            run_as_root=run_as_root
        )


@dataclass
class Service(KubernetesResource):

    @classmethod
    def from_api_object(cls, service):

        return cls(
            kind="Service",
            name=service.metadata.name,
            namespace=service.metadata.namespace,
            data={
                "type": service.spec.type,
                "selector": service.spec.selector or {}
            }
        )


@dataclass
class NetworkPolicy(KubernetesResource):

    @classmethod
    def from_api_object(cls, policy):

        return cls(
            kind="NetworkPolicy",
            name=policy.metadata.name,
            namespace=policy.metadata.namespace,
            data={
                "pod_selector": (
                    policy.spec.pod_selector.match_labels or {}
                ),
                "policy_types": (
                    policy.spec.policy_types or []
                )
            }
        )

Writing app/resources.py


### 3.2 `app/cluster.py`
`KubernetesCluster` reads pods, services and network policies from the Kubernetes API (all namespaces).

In [4]:
%%writefile app/cluster.py
from kubernetes import client, config

from .resources import (
    Pod,
    Service,
    NetworkPolicy
)


class KubernetesCluster:

    def __init__(self):

        config.load_kube_config()

        self.core_api = client.CoreV1Api()
        self.networking_api = client.NetworkingV1Api()

    def get_pods(self):

        response = (
            self.core_api
            .list_pod_for_all_namespaces()
        )

        return [
            Pod.from_api_object(pod)
            for pod in response.items
        ]

    def get_services(self):

        response = (
            self.core_api
            .list_service_for_all_namespaces()
        )

        return [
            Service.from_api_object(service)
            for service in response.items
        ]

    def get_network_policies(self):

        response = (
            self.networking_api
            .list_network_policy_for_all_namespaces()
        )

        return [
            NetworkPolicy.from_api_object(policy)
            for policy in response.items
        ]

    def get_resources(self):

        resources = []

        resources.extend(self.get_pods())
        resources.extend(self.get_services())
        resources.extend(self.get_network_policies())

        return resources

Writing app/cluster.py


## 4. Snapshot & baseline

### 4.1 `app/snapshot.py`
`ConfigurationSnapshot` normalizes the collected resources, sorts them, computes a SHA-256 configuration hash and saves the snapshot as JSON.

In [5]:
%%writefile app/snapshot.py
import hashlib
import json
from datetime import datetime


class ConfigurationSnapshot:

    def __init__(self, resources):

        self.timestamp = datetime.now().isoformat()

        self.resources = self.normalize(resources)

        self.configuration_hash = self.calculate_hash()

    def normalize(self, resources):

        normalized = []

        for resource in resources:

            item = {
                "kind": resource.kind,
                "name": resource.name,
                "namespace": resource.namespace,
                "data": resource.data
            }

            if resource.kind == "Pod":

                item["privileged"] = resource.privileged
                item["run_as_root"] = resource.run_as_root

            normalized.append(item)

        return sorted(
            normalized,
            key=lambda x: (
                x["kind"],
                x["namespace"],
                x["name"]
            )
        )

    def calculate_hash(self):

        serialized = json.dumps(
            self.resources,
            sort_keys=True
        )

        return hashlib.sha256(
            serialized.encode()
        ).hexdigest()

    def save(self, filename):

        data = {
            "timestamp": self.timestamp,
            "configuration_hash": self.configuration_hash,
            "resources": self.resources
        }

        with open(filename, "w") as file:

            json.dump(
                data,
                file,
                indent=2
            )

Writing app/snapshot.py


### 4.2 `app/baseline.py`
`BaselineManager` stores a snapshot as the trusted baseline and loads it back.

In [6]:
%%writefile app/baseline.py
import json


class BaselineManager:

    BASELINE_FILE = "snapshots/baseline.json"

    def create_baseline(self, snapshot):

        snapshot.save(self.BASELINE_FILE)

        print("Baseline created.")
        print(
            "Baseline hash:",
            snapshot.configuration_hash
        )

    def load_baseline(self):

        with open(self.BASELINE_FILE) as file:

            return json.load(file)

Writing app/baseline.py


## 5. Drift detection

### 5.1 `app/drift.py`
`DriftDetectionEngine` compares baseline and current snapshots and emits one `DriftEvent` per added, removed or modified resource field.

In [7]:
%%writefile app/drift.py
from dataclasses import dataclass
from typing import Any
import json


@dataclass
class DriftEvent:

    resource_kind: str
    resource_name: str
    namespace: str

    field: str

    change_type: str

    old_value: Any
    new_value: Any


class DriftDetectionEngine:

    def load_snapshot(self, filename):

        with open(filename) as file:

            return json.load(file)

    def index_resources(self, resources):

        return {
            (
                resource["kind"],
                resource["namespace"],
                resource["name"]
            ): resource

            for resource in resources
        }

    def flatten(self, resource):

        result = {}

        for key, value in resource.items():

            if key != "data":

                result[key] = value

        for key, value in resource.get(
            "data", {}
        ).items():

            result[key] = value

        return result

    def detect_drift(self, baseline, current):

        events = []

        baseline_resources = self.index_resources(
            baseline["resources"]
        )

        current_resources = self.index_resources(
            current["resources"]
        )

        all_keys = (
            set(baseline_resources)
            | set(current_resources)
        )

        for key in all_keys:

            old = baseline_resources.get(key)
            new = current_resources.get(key)

            # Resource added
            if old is None:

                events.append(
                    DriftEvent(
                        key[0],
                        key[2],
                        key[1],
                        "__resource__",
                        "added",
                        None,
                        new
                    )
                )

                continue

            # Resource removed
            if new is None:

                events.append(
                    DriftEvent(
                        key[0],
                        key[2],
                        key[1],
                        "__resource__",
                        "removed",
                        old,
                        None
                    )
                )

                continue

            old_fields = self.flatten(old)
            new_fields = self.flatten(new)

            fields = (
                set(old_fields)
                | set(new_fields)
            )

            for field in fields:

                old_value = old_fields.get(field)
                new_value = new_fields.get(field)

                if old_value != new_value:

                    events.append(
                        DriftEvent(
                            key[0],
                            key[2],
                            key[1],
                            field,
                            "modified",
                            old_value,
                            new_value
                        )
                    )

        return events

Writing app/drift.py


## 6. Rule-based security evaluation

### 6.1 `rules/security_rules.yaml`
Security rules: each rule matches a drift event by resource kind, field and (optionally) change type / old / new value, and carries an impact score.

In [8]:
%%writefile rules/security_rules.yaml
rules:

  - id: PRIVILEGED_CONTAINER
    description: Privileged container enabled
    resource_kind: Pod
    field: privileged
    old_value: false
    new_value: true
    impact: 30

  - id: ROOT_CONTAINER
    description: Container running as root
    resource_kind: Pod
    field: run_as_root
    old_value: false
    new_value: true
    impact: 20

  - id: NETWORK_POLICY_REMOVED
    description: NetworkPolicy removed
    resource_kind: NetworkPolicy
    field: __resource__
    change_type: removed
    impact: 25

  - id: SERVICE_LOADBALANCER
    description: Service exposed through LoadBalancer
    resource_kind: Service
    field: type
    new_value: LoadBalancer
    impact: 20

Writing rules/security_rules.yaml


### 6.2 `app/rules.py`
`SecurityRuleEngine` loads the YAML rules, matches drift events against them and sums the impact score.

In [9]:
%%writefile app/rules.py
import yaml


class SecurityRuleEngine:

    def __init__(self, rule_file):

        with open(rule_file) as file:

            data = yaml.safe_load(file)

        self.rules = data["rules"]

    def matches(self, event, rule):

        if event.resource_kind != rule["resource_kind"]:
            return False

        if event.field != rule["field"]:
            return False

        if "change_type" in rule:

            if event.change_type != rule["change_type"]:
                return False

        if "old_value" in rule:

            if event.old_value != rule["old_value"]:
                return False

        if "new_value" in rule:

            if event.new_value != rule["new_value"]:
                return False

        return True

    def evaluate(self, events):

        matches = []

        for event in events:

            for rule in self.rules:

                if self.matches(event, rule):

                    matches.append(
                        {
                            "rule_id": rule["id"],
                            "description": rule["description"],
                            "impact": rule["impact"],
                            "resource": (
                                f"{event.resource_kind}/"
                                f"{event.resource_name}"
                            ),
                            "field": event.field
                        }
                    )

        return matches

    def calculate_impact(self, matches):

        return sum(
            match["impact"]
            for match in matches
        )

Writing app/rules.py


## 7. Entry point & sample data

### 7.1 `app/main.py`
End-to-end run against a **live cluster** (needs a kubeconfig, so it is only written here, not executed in Colab).

In [10]:
%%writefile app/main.py
from .cluster import KubernetesCluster
from .snapshot import ConfigurationSnapshot
from .drift import DriftDetectionEngine
from .rules import SecurityRuleEngine


def main():

    cluster = KubernetesCluster()

    resources = cluster.get_resources()

    current_snapshot = ConfigurationSnapshot(
        resources
    )

    current_snapshot.save(
        "snapshots/current.json"
    )

    engine = DriftDetectionEngine()

    baseline = engine.load_snapshot(
        "snapshots/baseline.json"
    )

    current = engine.load_snapshot(
        "snapshots/current.json"
    )

    events = engine.detect_drift(
        baseline,
        current
    )

    print("\nDRIFT EVENTS")
    print("=" * 60)

    if not events:

        print("No configuration drift detected.")

    else:

        for event in events:

            print(
                f"\nResource: "
                f"{event.resource_kind}/"
                f"{event.resource_name}"
            )

            print(
                f"Namespace: "
                f"{event.namespace}"
            )

            print(
                f"Field: "
                f"{event.field}"
            )

            print(
                f"Change type: "
                f"{event.change_type}"
            )

            print(
                f"Old value: "
                f"{event.old_value}"
            )

            print(
                f"New value: "
                f"{event.new_value}"
            )

    rule_engine = SecurityRuleEngine(
        "rules/security_rules.yaml"
    )

    matches = rule_engine.evaluate(events)

    impact = rule_engine.calculate_impact(matches)

    print("\nSECURITY ASSESSMENT")
    print("=" * 60)

    for match in matches:

        print(
            f"\nRule: {match['rule_id']}"
        )

        print(
            f"Description: "
            f"{match['description']}"
        )

        print(
            f"Resource: "
            f"{match['resource']}"
        )

        print(
            f"Impact: "
            f"+{match['impact']}"
        )

    print(
        f"\nTotal Impact Score: {impact}"
    )


if __name__ == "__main__":
    main()

Writing app/main.py


### 7.2 `snapshots/baseline.json`
Sample baseline snapshot captured from the project's local cluster.

In [11]:
%%writefile snapshots/baseline.json
{
  "timestamp": "2026-09-21T15:21:57.731465",
  "configuration_hash": "953ebf1a357d164f2b8a1f74ba958e1a94dfea52f8db82bd7ba8d64526d4f46c",
  "resources": [
    {
      "kind": "NetworkPolicy",
      "name": "demo-policy",
      "namespace": "security-demo",
      "data": {
        "pod_selector": {
          "app": "demo"
        },
        "policy_types": [
          "Ingress"
        ]
      }
    },
    {
      "kind": "Pod",
      "name": "coredns-589f44dc88-8nzsh",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "k8s-app": "kube-dns",
          "pod-template-hash": "589f44dc88"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "coredns-589f44dc88-nblr4",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "k8s-app": "kube-dns",
          "pod-template-hash": "589f44dc88"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "etcd-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "etcd",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kindnet-28gbr",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "app": "kindnet",
          "controller-revision-hash": "5db9b784b5",
          "k8s-app": "kindnet",
          "pod-template-generation": "1",
          "tier": "node"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-apiserver-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-apiserver",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-controller-manager-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-controller-manager",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-proxy-7m8px",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "controller-revision-hash": "88bb97857",
          "k8s-app": "kube-proxy",
          "pod-template-generation": "1"
        }
      },
      "privileged": true,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-scheduler-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-scheduler",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "local-path-provisioner-855c7b7774-qqspm",
      "namespace": "local-path-storage",
      "data": {
        "labels": {
          "app": "local-path-provisioner",
          "pod-template-hash": "855c7b7774"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "demo-pod",
      "namespace": "security-demo",
      "data": {
        "labels": {
          "app": "demo"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Service",
      "name": "kubernetes",
      "namespace": "default",
      "data": {
        "type": "ClusterIP",
        "selector": {}
      }
    },
    {
      "kind": "Service",
      "name": "kube-dns",
      "namespace": "kube-system",
      "data": {
        "type": "ClusterIP",
        "selector": {
          "k8s-app": "kube-dns"
        }
      }
    },
    {
      "kind": "Service",
      "name": "demo-service",
      "namespace": "security-demo",
      "data": {
        "type": "ClusterIP",
        "selector": {
          "app": "demo"
        }
      }
    }
  ]
}

Writing snapshots/baseline.json


### 7.3 `snapshots/current.json`
Sample current snapshot — `demo-pod` in namespace `security-demo` is now privileged.

In [12]:
%%writefile snapshots/current.json
{
  "timestamp": "2026-09-21T15:28:24.466024",
  "configuration_hash": "395275bbded4af1b0fe651318f35f77ce8d0ad09b2317e0d7c8a6cc81f226a6f",
  "resources": [
    {
      "kind": "NetworkPolicy",
      "name": "demo-policy",
      "namespace": "security-demo",
      "data": {
        "pod_selector": {
          "app": "demo"
        },
        "policy_types": [
          "Ingress"
        ]
      }
    },
    {
      "kind": "Pod",
      "name": "coredns-589f44dc88-8nzsh",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "k8s-app": "kube-dns",
          "pod-template-hash": "589f44dc88"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "coredns-589f44dc88-nblr4",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "k8s-app": "kube-dns",
          "pod-template-hash": "589f44dc88"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "etcd-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "etcd",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kindnet-28gbr",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "app": "kindnet",
          "controller-revision-hash": "5db9b784b5",
          "k8s-app": "kindnet",
          "pod-template-generation": "1",
          "tier": "node"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-apiserver-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-apiserver",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-controller-manager-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-controller-manager",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-proxy-7m8px",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "controller-revision-hash": "88bb97857",
          "k8s-app": "kube-proxy",
          "pod-template-generation": "1"
        }
      },
      "privileged": true,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "kube-scheduler-desktop-control-plane",
      "namespace": "kube-system",
      "data": {
        "labels": {
          "component": "kube-scheduler",
          "tier": "control-plane"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "local-path-provisioner-855c7b7774-qqspm",
      "namespace": "local-path-storage",
      "data": {
        "labels": {
          "app": "local-path-provisioner",
          "pod-template-hash": "855c7b7774"
        }
      },
      "privileged": false,
      "run_as_root": false
    },
    {
      "kind": "Pod",
      "name": "demo-pod",
      "namespace": "security-demo",
      "data": {
        "labels": {
          "app": "demo"
        }
      },
      "privileged": true,
      "run_as_root": false
    },
    {
      "kind": "Service",
      "name": "kubernetes",
      "namespace": "default",
      "data": {
        "type": "ClusterIP",
        "selector": {}
      }
    },
    {
      "kind": "Service",
      "name": "kube-dns",
      "namespace": "kube-system",
      "data": {
        "type": "ClusterIP",
        "selector": {
          "k8s-app": "kube-dns"
        }
      }
    },
    {
      "kind": "Service",
      "name": "demo-service",
      "namespace": "security-demo",
      "data": {
        "type": "ClusterIP",
        "selector": {
          "app": "demo"
        }
      }
    }
  ]
}

Writing snapshots/current.json


## 8. Run the pipeline (demo mode)
Same drift-detection and rule-evaluation steps as `main.py`, but reading the saved snapshots instead of calling the cluster.

In [13]:
from app.drift import DriftDetectionEngine
from app.rules import SecurityRuleEngine

engine = DriftDetectionEngine()
baseline = engine.load_snapshot("snapshots/baseline.json")
current  = engine.load_snapshot("snapshots/current.json")

print(f"Baseline resources: {len(baseline['resources'])}   |   Current resources: {len(current['resources'])}")
print(f"Baseline hash: {baseline['configuration_hash'][:16]}...   Current hash: {current['configuration_hash'][:16]}...")

events = engine.detect_drift(baseline, current)

print("\nDRIFT EVENTS")
print("=" * 60)
if not events:
    print("No configuration drift detected.")
for e in events:
    print(f"\nResource: {e.resource_kind}/{e.resource_name}")
    print(f"Namespace: {e.namespace}")
    print(f"Field: {e.field}")
    print(f"Change type: {e.change_type}")
    print(f"Old value: {e.old_value}")
    print(f"New value: {e.new_value}")

rule_engine = SecurityRuleEngine("rules/security_rules.yaml")
matches = rule_engine.evaluate(events)
impact = rule_engine.calculate_impact(matches)

print("\nSECURITY ASSESSMENT")
print("=" * 60)
for m in matches:
    print(f"\nRule: {m['rule_id']}")
    print(f"Description: {m['description']}")
    print(f"Resource: {m['resource']}")
    print(f"Impact: +{m['impact']}")
print(f"\nTotal Impact Score: {impact}")

Baseline resources: 14   |   Current resources: 14
Baseline hash: 953ebf1a357d164f...   Current hash: 395275bbded4af1b...

DRIFT EVENTS

Resource: Pod/demo-pod
Namespace: security-demo
Field: privileged
Change type: modified
Old value: False
New value: True

SECURITY ASSESSMENT

Rule: PRIVILEGED_CONTAINER
Description: Privileged container enabled
Resource: Pod/demo-pod
Impact: +30

Total Impact Score: 30


## 9. Try your own change (optional)
Edit the cell below to flip a field in the *current* snapshot (for example make `demo-pod` run as root) and re-run section 8's logic to see a different rule fire.

In [14]:
import json
from app.drift import DriftDetectionEngine
from app.rules import SecurityRuleEngine

with open("snapshots/current.json") as f:
    snap = json.load(f)

# Example change: also mark demo-pod as running as root
for r in snap["resources"]:
    if r["kind"] == "Pod" and r["name"] == "demo-pod":
        r["run_as_root"] = True

with open("snapshots/current_modified.json", "w") as f:
    json.dump(snap, f, indent=2)

engine = DriftDetectionEngine()
events = engine.detect_drift(engine.load_snapshot("snapshots/baseline.json"),
                             engine.load_snapshot("snapshots/current_modified.json"))
rules = SecurityRuleEngine("rules/security_rules.yaml")
matches = rules.evaluate(events)
for m in matches:
    print(f"{m['rule_id']:<24} {m['resource']:<22} +{m['impact']}")
print("Total Impact Score:", rules.calculate_impact(matches))

PRIVILEGED_CONTAINER     Pod/demo-pod           +30
ROOT_CONTAINER           Pod/demo-pod           +20
Total Impact Score: 50
